# Person 4: Transformer sequential model (self-attention over MFCC frame sequences)

This notebook runs `scripts/run_person4_sequential.py`. All model code lives in `src/models/neural.py` (the `SequentialTransformerClassifier` and its training loop) and `src/preprocessing/sequences.py` (shared with Person 2). The notebook does not reimplement loading, the split, or preprocessing.

**Why a Transformer, not another recurrent model:** Person 2's model is a BiGRU/BiLSTM family -- recurrent, left-to-right state. This model attends over the whole frame sequence at once (self-attention + fixed positional encoding), so the architectural comparison in the report is about the inductive bias itself, not a hyperparameter variant of the same family.

**Rules followed:** only `data/splits/shared_split.csv` is used (70/15/15, seed 42). Every tuning and early-stopping decision uses the validation split. The internal test split (`split == "test"`) is evaluated once, after the configuration is locked. The official Zindi `Test.csv` is never used. The acoustic front end (13 MFCC, 8 s fixed window) matches Person 2's and the classical baselines, so architecture is the only thing being varied.

**Runtime:** use a GPU runtime (*Runtime -> Change runtime type -> T4 GPU*). A full run takes roughly 1-2 hours on a T4 (five search stages + lock + two extra seeds). On a CPU it is much slower; use the `--quick` smoke test there.

**Before running in Colab:** upload `Swahili_words.zip` and `Train.csv` (plus `Test.csv` if you have it) to one Google Drive folder, and set `DRIVE_DATA_DIR` below to that folder.

In [ ]:
# --- Settings ---------------------------------------------------------------
REPO_URL = "https://github.com/Orrie-Dan/NLP-and-Language-Technologies.git"
BRANCH = "person4"                                        # replace with your actual branch name
DRIVE_DATA_DIR = "/content/drive/MyDrive/swahili_audio"    # folder holding the data files
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/swahili_audio/person4_results"

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/NLP-and-Language-Technologies")
    if not repo.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    raw = repo / "data" / "raw"
    data_dir = Path(DRIVE_DATA_DIR)
    if not (data_dir / "Swahili_words.zip").exists():
        # DRIVE_DATA_DIR is wrong or empty: look for the zip anywhere in My Drive.
        print(f"Swahili_words.zip not in {data_dir}; searching My Drive...")
        found = next(Path("/content/drive/MyDrive").rglob("Swahili_words.zip"), None)
        if found is None:
            raise FileNotFoundError(
                "Swahili_words.zip is not in your Google Drive. Upload Swahili_words.zip and "
                "Train.csv to one Drive folder, set DRIVE_DATA_DIR to it, and rerun this cell.")
        data_dir = found.parent
        print("Using data folder:", data_dir)
    # Only the zip and Train.csv are needed; Test.csv and SampleSubmission.csv are optional.
    for name in ["Swahili_words.zip", "Train.csv", "Test.csv", "SampleSubmission.csv"]:
        source = data_dir / name
        if (raw / name).exists():
            continue
        if source.exists():
            shutil.copy(source, raw / name)
        elif name in ("Swahili_words.zip", "Train.csv"):
            raise FileNotFoundError(f"{source} is required. Put it next to Swahili_words.zip.")
else:
    # Running locally from notebooks/: move to the repository root.
    if Path.cwd().name == "notebooks":
        os.chdir("..")
print("Working directory:", Path.cwd())
print(sorted(p.name for p in Path("data/raw").iterdir()))

In [ ]:
%pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## Run the experiment

The first time, this extracts 13-coefficient MFCC frame sequences for all 4,200 labelled clips (cached in `results/models/transformer/cache/`). It then:

1. **Validation search.** One factor changes at a time and the best validation macro-F1 is carried forward: input features (static MFCC vs MFCC + Δ + ΔΔ), pooling (masked mean vs a learned CLS token), model capacity (`d_model` 64/128/256), depth (1/2/4 encoder layers), and dropout (0.1/0.3/0.5).
2. **Locked test evaluation.** The selected model is evaluated once on the internal test split.
3. **Seed robustness.** The same locked configuration is retrained with seeds 43 and 44.

In [ ]:
!{sys.executable} scripts/run_person4_sequential.py --device auto

## Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

out = Path("results/models/transformer")
fig = Path("results/figures/transformer")
pd.set_option("display.precision", 4)

print("Validation search (all runs):")
display(pd.read_csv(out / "validation_experiments.csv")[
    ["stage", "run", "same_as", "d_model", "num_heads", "num_layers", "ffn_dim",
     "pooling", "feature_streams", "dropout", "best_epoch", "n_parameters", "val_accuracy", "val_macro_f1"]])

print("Selected configuration:")
print(json.dumps(json.loads((out / "selected_config.json").read_text()), indent=2))

In [ ]:
final = json.loads((out / "final_transformer_results.json").read_text())
classical = pd.read_csv("results/classical/final_classical_results.csv")
metric_cols = ["model", "accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall", "macro_roc_auc"]
# The classical results have no ROC-AUC column; it shows as NaN for those rows.
comparison_rows = [classical.reindex(columns=metric_cols)]
recurrent_final = Path("results/models/recurrent/final_recurrent_results.csv")
if recurrent_final.is_file():
    comparison_rows.append(pd.read_csv(recurrent_final).reindex(columns=metric_cols))
comparison_rows.append(pd.DataFrame([{"model": f"transformer ({final['selected_run']})", **final["test"]}]))
comparison = pd.concat(comparison_rows, ignore_index=True)
print("Internal test comparison:")
display(comparison)
print("Seed robustness (locked configuration):")
display(pd.read_csv(out / "seed_robustness.csv"))
print("Accuracy by clip duration (internal test):")
display(pd.read_csv(out / "test_accuracy_by_duration.csv"))

In [ ]:
for name in ["validation_experiments", "learning_curves", "test_confusion_matrix", "test_roc_curves",
             "test_per_class_f1_vs_classical", "test_accuracy_by_duration"]:
    print(name)
    display(Image(filename=str(fig / f"{name}.png"), width=720))

## Save results to Drive (Colab only)

The Colab disk is wiped when the session ends. This copies the results and figures to Drive, without the feature cache or checkpoints. Download the folder, then copy `results/models/transformer/` and `results/figures/transformer/` into the repository and commit them.

In [ ]:
if IN_COLAB:
    dest = Path(DRIVE_RESULTS_DIR)
    for src in [out, fig]:
        target = dest / src
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(src, target, ignore=shutil.ignore_patterns("cache", "checkpoints"))
    print("Saved to", dest)